**Import Required Libraries**

In [1]:
# Install if needed (Colab has most pre-installed)
!pip install pandas numpy tensorflow scikit-learn

import pandas as pd
import numpy as np
import re
import string
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("✅ Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

✅ Libraries imported successfully!
TensorFlow version: 2.19.0


**Load Dataset**

In [2]:
from google.colab import files
files.upload()

Saving Email_spam_ham.xlsx to Email_spam_ham.xlsx


{'Email_spam_ham.xlsx': b'PK\x03\x04\x14\x00\x06\x00\x08\x00\x00\x00!\x00b\xee\x9dh^\x01\x00\x00\x90\x04\x00\x00\x13\x00\x08\x02[Content_Types].xml \xa2\x04\x02(\xa0\x00\x02\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x0

**Text Cleaning & Preprocessing**

In [3]:
import pandas as pd
import re

# 1. Load your EXCEL file (not CSV)
df = pd.read_excel('Email_spam_ham.xlsx')  # ✅ Excel file

# 2. Check actual column names
print("✅ Available columns:", df.columns.tolist())

# 3. Clean based on actual structure
if 'text' in df.columns and 'body' in df.columns:
    # Case: You have BOTH 'text' (subject) and 'body'
    def clean_full_email(subject, body):
        full = str(subject) + " " + str(body)
        full = full.lower()
        full = re.sub(r'http\S+|www\S+|https\S+', '', full, flags=re.MULTILINE)
        full = re.sub(r'\S+@\S+', '', full)
        full = re.sub(r'\s+', ' ', full).strip()
        return full

    df['cleaned_text'] = df.apply(lambda row: clean_full_email(row['text'], row['body']), axis=1)
    X = df['cleaned_text']
    y = df['label_num']

elif 'text' in df.columns:
    # Case: You have ONLY 'text' (subject or full email)
    def clean_text(text):
        if pd.isna(text):
            return ""
        text = str(text).lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        text = re.sub(r'\S+@\S+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    df['cleaned_text'] = df['text'].apply(clean_text)
    X = df['cleaned_text']
    y = df['label_num']

else:
    raise ValueError("❌ Expected columns like 'text' (and optionally 'body') not found!")

print("✅ Cleaned sample:", X.iloc[0])

✅ Available columns: ['Unnamed: 0', 'label', 'text', 'body', 'label_num']
✅ Cleaned sample: enron methanol ; meter # : 988291 this is a follow up to the note i gave you on monday , 4 / 3 / 00 { preliminary_x000d_ flow data provided by daren } ._x000d_ please override pop ' s daily volume { presently zero } to reflect daily_x000d_ activity you can obtain from gas control ._x000d_ this change is needed asap for economics purposes .


**Email Count Tracking**

In [4]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

# --------------------------------------------------
# STEP 1: Load & Clean Data (Your Code – Perfect!)
# --------------------------------------------------
df_raw = pd.read_excel('Email_spam_ham.xlsx')
print(f"📧 Total emails BEFORE cleaning: {len(df_raw)}")

duplicate_emails_raw = df_raw.duplicated(subset=['text']).sum()
print(f"🔁 Duplicates in raw data: {duplicate_emails_raw}")

df_cleaned = df_raw.dropna(subset=['text', 'label_num']).copy()
print(f"🧹 Emails after removing missing text/label: {len(df_cleaned)}")

df_cleaned = df_cleaned.drop_duplicates(subset=['text', 'label_num']).reset_index(drop=True)
print(f"🧼 Emails after deduplication: {len(df_cleaned)}")

def clean_text(text):
    if isinstance(text, str):
        text = text.lower()
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        text = re.sub(r'\S+@\S+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    return ""

df_cleaned['cleaned_text'] = df_cleaned['text'].apply(clean_text)
df_final = df_cleaned[df_cleaned['cleaned_text'].str.strip() != ''].reset_index(drop=True)
print(f"✅ Final usable emails AFTER full cleaning: {len(df_final)}")

print("\n📊 Final label distribution:")
print(df_final['label_num'].value_counts().sort_index())

📧 Total emails BEFORE cleaning: 5185
🔁 Duplicates in raw data: 866
🧹 Emails after removing missing text/label: 5109
🧼 Emails after deduplication: 4307
✅ Final usable emails AFTER full cleaning: 4290

📊 Final label distribution:
label_num
0.0    2947
1.0    1343
Name: count, dtype: int64


In [5]:
# STEP 2: Prepare X and y (CRITICAL: Use df_final!)
# --------------------------------------------------
X = df_final['cleaned_text']      # ✅ Cleaned full text (subject or subject+body)
y = df_final['label_num']        # ✅ 0 = ham, 1 = spam


**Split Data (Train/Val/Test)**

In [6]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 3003 | Val: 643 | Test: 644


**Tokenization & Padding (TensorFlow)**

In [7]:
MAX_WORDS = 20000   # Vocabulary size
MAX_LEN = 500       # Sequence length

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)  # Fit only on train!

# Convert to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print("✅ Tokenization & padding complete!")
print(f"Training input shape: {X_train_pad.shape}")

✅ Tokenization & padding complete!
Training input shape: (3003, 500)


**Build CNN Model**

In [10]:
# Define hyperparameters
from tensorflow.keras.layers import Concatenate

EMBEDDING_DIM = 128

# Multi-kernel CNN for n-gram detection
input_layer = Input(shape=(MAX_LEN,), name='input')
embedding = Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN)(input_layer)

conv1 = Conv1D(filters=64, kernel_size=3, activation='relu')(embedding)
conv2 = Conv1D(filters=64, kernel_size=4, activation='relu')(embedding)
conv3 = Conv1D(filters=64, kernel_size=5, activation='relu')(embedding)

pool1 = GlobalMaxPooling1D()(conv1)
pool2 = GlobalMaxPooling1D()(conv2)
pool3 = GlobalMaxPooling1D()(conv3)

concat = Concatenate()([pool1, pool2, pool3])

dense = Dense(64, activation='relu')(concat)
dropout = Dropout(0.5)(dense)
output = Dense(1, activation='sigmoid', name='output')(dropout)

model = Model(inputs=input_layer, outputs=output)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 'Precision', 'Recall']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 500)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 500, 128)  │  2,560,000 │ input[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 498, 64)   │     24,640 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, 497, 64)   │     32,832 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 496, 64)   │     41,024 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d_6[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d_7[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d_8[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 192)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │     12,352 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         65 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,670,913 (10.19 MB)

 Trainable params: 2,670,913 (10.19 MB)

 Non-trainable params: 0 (0.00 B)

**Train with Early Stopping & Class Weighting**

In [11]:
# Compute class weights (optional but recommended)
classes = np.array([0, 1])
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print("✅ Class weights:", class_weight_dict)

# Callbacks
early_stop = EarlyStopping(
    monitor='val_Precision',  # ← Capital 'P'
    mode='max',
    patience=3,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=0
)

# Train!
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weight_dict,
    verbose=1
)

✅ Class weights: {0: np.float64(0.7278235579253515), 1: np.float64(1.5973404255319148)}
Epoch 1/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 11s 122ms/step - Precision: 0.4627 - Recall: 0.6587 - accuracy: 0.6468 - loss: 0.6491 - val_Precision: 0.7557 - val_Recall: 0.8308 - val_accuracy: 0.8631 - val_loss: 0.3500 - learning_rate: 0.0010
Epoch 2/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - Precision: 0.8346 - Recall: 0.9571 - accuracy: 0.9282 - loss: 0.2202 - val_Precision: 0.8419 - val_Recall: 0.9005 - val_accuracy: 0.9160 - val_loss: 0.2061 - learning_rate: 0.0010
Epoch 3/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - Precision: 0.9505 - Recall: 0.9872 - accuracy: 0.9813 - loss: 0.0545 - val_Precision: 0.8493 - val_Recall: 0.9254 - val_accuracy: 0.9253 - val_loss: 0.1958 - learning_rate: 0.0010
Epoch 4/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - Precision: 0.9815 - Recall: 0.9990 - accuracy: 0.9940 - loss: 0.0243 - val_Precision: 0.8958 - val_Recall: 0.8557 - val_accuracy: 0.9238 - val_loss: 0.2121 - 

**Evaluate**

In [12]:
# Evaluate on truly unseen test set
test_results = model.evaluate(X_test_pad, y_test, verbose=0)
test_acc, test_prec, test_rec = test_results[1], test_results[2], test_results[3]

print(f"\n🎯 FINAL TEST SET RESULTS:")
print(f"✅ Accuracy:  {test_acc:.4f}")
print(f"🔍 Precision: {test_prec:.4f}  → False Positives: {100*(1 - test_prec):.2f}%")
print(f"🛡️ Recall:    {test_rec:.4f}")


🎯 FINAL TEST SET RESULTS:
✅ Accuracy:  0.9177
🔍 Precision: 0.8860  → False Positives: 11.40%
🛡️ Recall:    0.8465


**Save Model**

In [13]:
import pickle
import tensorflow as tf
from google.colab import files
import shutil

# 1. Save the trained model (HDF5 format)
model.save('spam_email_model.h5')
print("✅ Model saved as 'spam_email_model.h5'")

# 2. Save the tokenizer
with open('tokenizer.pickle', 'wb') as f:
    pickle.dump(tokenizer, f)
print("✅ Tokenizer saved as 'tokenizer.pickle'")

# 3. Create a ZIP file containing both
shutil.make_archive('anti_spamx_email_model', 'zip', '.', 'spam_email_model.h5')
shutil.make_archive('anti_spamx_email_model', 'zip', '.', 'tokenizer.pickle')

# Alternative: Add both to one zip
import zipfile
with zipfile.ZipFile('anti_spamx_email_model.zip', 'w') as zipf:
    zipf.write('spam_email_model.h5')
    zipf.write('tokenizer.pickle')

print("📦 ZIP archive created: anti_spamx_email_model.zip")

# 4. Download directly to your computer
files.download('anti_spamx_email_model.zip')

✅ Model saved as 'spam_email_model.h5'
✅ Tokenizer saved as 'tokenizer.pickle'
📦 ZIP archive created: anti_spamx_email_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>